# FPL Model Comparison: 2024-25 Season

Train and compare all available FPL models on the 2024-25 season:

| Model | Predictor | Optimizer | Transfers? | Mid-Season Retrain? |
|-------|-----------|-----------|------------|---------------------|
| **form-greedy** | Recent form average | Greedy lineup | No | No |
| **xgb-greedy** | XGBoost (25+ features) | Greedy lineup + transfers | Yes | No |
| **xgb-lp** | XGBoost | Integer Linear Program | Yes | No |
| **sequence-lp** | LSTM sequence model | Integer Linear Program | Yes | No |
| **ppo-agent** | End-to-end RL (PPO) | Policy network | No (lineup/captain only) | No |
| **xgb-greedy-retrain** | XGBoost (recency decay 0.95) | Greedy lineup + transfers | Yes | Every 5 GWs |

## 1. Setup — Ingest Data

In [ ]:
import asyncio
from pathlib import Path

from fpl_model.data.ingest import Ingester

# Seasons: train on 2023-24, evaluate on 2024-25
TRAIN_SEASONS = ["2021-22", "2022-23", "2023-24"]
EVAL_SEASON = "2024-25"
ALL_SEASONS = TRAIN_SEASONS + [EVAL_SEASON]

DB_PATH = Path("../data/fpl.db")
CACHE_DIR = Path("../data/cache")

ingester = Ingester(db_path=DB_PATH, cache_dir=CACHE_DIR)
db = ingester.db

print(f"Ingesting seasons: {ALL_SEASONS}")
await ingester.ingest_seasons(ALL_SEASONS)

# Verify data loaded
for table in ["players", "gameweek_performances", "fixtures", "teams"]:
    for season in ALL_SEASONS:
        df = db.read(table, where={"season": season})
        print(f"  {table} ({season}): {len(df)} rows")

## 2. Build Training Data

Load historical data into the format expected by model `train()` methods.

In [ ]:
from fpl_model.models.base import HistoricalData, SeasonData

def load_season_data(db, season):
    """Load a SeasonData object from the database."""
    gw_perf = db.read("gameweek_performances", where={"season": season})
    players = db.read("players", where={"season": season})
    fixtures = db.read("fixtures", where={"season": season})
    teams = db.read("teams", where={"season": season})
    max_gw = int(gw_perf["gameweek"].max()) if len(gw_perf) > 0 else 1
    return SeasonData(
        gameweek_performances=gw_perf,
        fixtures=fixtures,
        players=players,
        teams=teams,
        current_gameweek=max_gw,
        season=season,
    )

# Build HistoricalData for training (includes all seasons)
historical = HistoricalData(seasons={})
for season in ALL_SEASONS:
    historical.seasons[season] = load_season_data(db, season)
    print(f"Loaded {season}: {len(historical.seasons[season].gameweek_performances)} GW rows, "
          f"{len(historical.seasons[season].players)} players")

## 3. Train Models

Train all models that require training. `form-greedy` is stateless (no training needed).

In [ ]:
import time
from fpl_model.models.defaults import get_default_registry, create_ppo_agent
from fpl_model.models.base import PredictOptimizeModel
from fpl_model.models.predictors.xgboost_predictor import XGBoostPredictor
from fpl_model.models.optimizers.greedy import GreedyOptimizer

registry = get_default_registry()

# Train the predict+optimize models that need it
trainable = ["xgb-greedy", "xgb-lp", "sequence-lp"]
for name in trainable:
    model = registry.get(name)
    print(f"Training {name}...", end=" ", flush=True)
    t0 = time.time()
    model.train(historical)
    elapsed = time.time() - t0
    print(f"done ({elapsed:.1f}s)")

# XGBoost with recency decay (will be retrained mid-season during simulation)
xgb_retrain = PredictOptimizeModel(
    XGBoostPredictor(recency_decay=0.95),
    GreedyOptimizer(enable_transfers=True),
)
print("Training xgb-greedy-retrain...", end=" ", flush=True)
t0 = time.time()
xgb_retrain.train(historical)
elapsed = time.time() - t0
print(f"done ({elapsed:.1f}s)")

# Train PPO agent separately (needs db + seasons)
print("Training ppo-agent...", end=" ", flush=True)
t0 = time.time()
ppo_agent = create_ppo_agent(
    db=db,
    seasons=ALL_SEASONS,
    hidden_size=64,
    train_epochs=3,
    episodes_per_update=2,
)
ppo_agent.train(historical)
elapsed = time.time() - t0
print(f"done ({elapsed:.1f}s)")

print("\nAll models trained.")

## 4. Simulate All Models on 2024-25

Run each model through the full 2024-25 season using `SeasonSimulator`.

In [ ]:
from fpl_model.simulation.engine import SeasonSimulator

# Collect all models to simulate
models = {name: registry.get(name) for name in registry.list()}
models["ppo-agent"] = ppo_agent
models["xgb-greedy-retrain"] = xgb_retrain

results = {}
for name, model in models.items():
    print(f"Simulating {name}...", end=" ", flush=True)
    t0 = time.time()
    # Use mid-season retraining for the retrain variant
    if name == "xgb-greedy-retrain":
        sim = SeasonSimulator(
            model=model,
            season=EVAL_SEASON,
            db=db,
            retrain_every_n_gws=5,
            train_seasons=TRAIN_SEASONS,
        )
    else:
        sim = SeasonSimulator(model=model, season=EVAL_SEASON, db=db)
    result = sim.run()
    elapsed = time.time() - t0
    results[name] = result
    print(f"{result.total_points} pts ({len(result.gameweek_points)} GWs, {elapsed:.1f}s)")

print(f"\nAll {len(results)} models simulated.")

## 5. Comparison Table

In [ ]:
import pandas as pd
from fpl_model.evaluation.comparison import compare_results
from fpl_model.evaluation.reports import format_comparison

comparison = compare_results(results)
print(format_comparison(comparison))

# Also show as a DataFrame for easier reading
comp_df = pd.DataFrame(comparison)
comp_df["best_gw"] = comp_df["best_gw"].apply(lambda x: f"GW{x[0]} ({x[1]}pts)")
comp_df["worst_gw"] = comp_df["worst_gw"].apply(lambda x: f"GW{x[0]} ({x[1]}pts)")
comp_df.index = range(1, len(comp_df) + 1)
comp_df.index.name = "rank"
comp_df

## 6. Gameweek-by-Gameweek Points Comparison

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(2, 1, figsize=(14, 10), sharex=True)

# --- Per-GW points ---
ax = axes[0]
for name, result in results.items():
    gws = sorted(result.gameweek_points.keys())
    pts = [result.gameweek_points[gw] for gw in gws]
    ax.plot(gws, pts, marker=".", markersize=4, label=name, alpha=0.8)
ax.set_ylabel("Points")
ax.set_title(f"Gameweek Points — {EVAL_SEASON}")
ax.legend(loc="upper right", fontsize=9)
ax.grid(True, alpha=0.3)

# --- Cumulative points ---
ax = axes[1]
for name, result in results.items():
    gws = sorted(result.gameweek_points.keys())
    cumulative = []
    total = 0
    for gw in gws:
        total += result.gameweek_points[gw]
        cumulative.append(total)
    ax.plot(gws, cumulative, marker=".", markersize=4, label=name, alpha=0.8)
ax.set_xlabel("Gameweek")
ax.set_ylabel("Cumulative Points")
ax.set_title(f"Cumulative Points — {EVAL_SEASON}")
ax.legend(loc="upper left", fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Transfer Activity & Costs

In [ ]:
from fpl_model.simulation.actions import Transfer

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Transfer hit costs per model
ax = axes[0]
names = []
costs = []
for name in sorted(results.keys()):
    result = results[name]
    total_cost = sum(result.transfer_costs.values())
    names.append(name)
    costs.append(total_cost)
ax.barh(names, costs, color="salmon")
ax.set_xlabel("Total Transfer Hit Cost (pts)")
ax.set_title("Transfer Point Hits")
ax.grid(True, alpha=0.3, axis="x")

# Number of transfers made per model
ax = axes[1]
transfer_counts = []
for name in sorted(results.keys()):
    result = results[name]
    count = sum(
        sum(1 for a in actions if isinstance(a, Transfer))
        for actions in result.actions_log.values()
    )
    transfer_counts.append(count)
ax.barh(names, transfer_counts, color="steelblue")
ax.set_xlabel("Total Transfers Made")
ax.set_title("Transfer Volume")
ax.grid(True, alpha=0.3, axis="x")

plt.tight_layout()
plt.show()

## 8. Points Distribution per Model

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

model_names = sorted(results.keys())
data_for_box = []
for name in model_names:
    gw_pts = list(results[name].gameweek_points.values())
    data_for_box.append(gw_pts)

bp = ax.boxplot(data_for_box, labels=model_names, patch_artist=True, vert=True)
colors = plt.cm.Set2.colors
for patch, color in zip(bp["boxes"], colors):
    patch.set_facecolor(color)

ax.set_ylabel("Points per Gameweek")
ax.set_title(f"GW Points Distribution — {EVAL_SEASON}")
ax.grid(True, alpha=0.3, axis="y")
plt.tight_layout()
plt.show()

## 9. Budget Trajectory

How each model's remaining budget evolves over the season (reflects transfer spending and player value changes).

In [ ]:
fig, ax = plt.subplots(figsize=(14, 5))

for name, result in results.items():
    if result.budget_history:
        gws = sorted(result.budget_history.keys())
        budgets = [result.budget_history[gw] / 10.0 for gw in gws]  # convert to millions
        ax.plot(gws, budgets, marker=".", markersize=4, label=name, alpha=0.8)

ax.set_xlabel("Gameweek")
ax.set_ylabel("Remaining Budget (£m)")
ax.set_title(f"Budget Over Season — {EVAL_SEASON}")
ax.legend(loc="best", fontsize=9)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Summary

The final ranking table with net points (total minus transfer hits).

In [ ]:
summary_rows = []
for row in comparison:
    net = row["total_points"]  # transfer costs already deducted in simulation scoring
    summary_rows.append({
        "Model": row["name"],
        "Total Points": row["total_points"],
        "Avg/GW": round(row["avg_points_per_gw"], 1),
        "Transfer Hits": row["total_transfer_cost"],
        "Best GW": f"GW{row['best_gw'][0]} ({row['best_gw'][1]}pts)",
        "Worst GW": f"GW{row['worst_gw'][0]} ({row['worst_gw'][1]}pts)",
        "GWs": row["num_gameweeks"],
    })

summary_df = pd.DataFrame(summary_rows)
summary_df.index = range(1, len(summary_df) + 1)
summary_df.index.name = "Rank"
summary_df

## 11. Gameweek-by-Gameweek Squad Details

For each model, show the starting XI (with formation), captain, transfers made, chips played, and points scored each gameweek. Use the dropdown to select a model.

In [ ]:
from collections import Counter
from IPython.display import display, Markdown
from fpl_model.simulation.actions import SetLineup, SetCaptain, SetViceCaptain, Transfer, PlayChip
from ipywidgets import interact, Dropdown

# Build player name/position/team lookups from the eval season
_players_df = db.read("players", where={"season": EVAL_SEASON})
_teams_df = db.read("teams", where={"season": EVAL_SEASON})
_team_map = dict(zip(_teams_df["team_code"], _teams_df["short_name"]))
_player_name = dict(zip(_players_df["code"], _players_df["web_name"]))
_player_pos = dict(zip(_players_df["code"], _players_df["element_type"]))
_player_team = dict(zip(_players_df["code"], _players_df["team_code"]))

POS_LABEL = {1: "GK", 2: "DEF", 3: "MID", 4: "FWD", 5: "MGR"}

def _fmt(code, cap=None, vc=None):
    name = _player_name.get(code, f"#{code}")
    team = _team_map.get(_player_team.get(code, 0), "?")
    tag = " **(C)**" if code == cap else (" *(V)*" if code == vc else "")
    return f"{name} ({team}){tag}"

def _formation(codes):
    c = Counter(_player_pos.get(p, 0) for p in codes)
    return f"{c.get(2,0)}-{c.get(3,0)}-{c.get(4,0)}"

def _group_by_pos(codes, cap=None, vc=None):
    groups = {pos: [] for pos in ["GK", "DEF", "MID", "FWD"]}
    for code in codes:
        pos = POS_LABEL.get(_player_pos.get(code, 0), "FWD")
        if pos in groups:
            groups[pos].append(_fmt(code, cap, vc))
    lines = []
    for pos in ["GK", "DEF", "MID", "FWD"]:
        if groups[pos]:
            lines.append(f"**{pos}:** {', '.join(groups[pos])}")
    return "  \n".join(lines)

def show_model_gws(model_name):
    result = results[model_name]
    for gw in sorted(result.gameweek_points.keys()):
        actions = result.actions_log.get(gw, [])
        pts = result.gameweek_points[gw]
        cost = result.transfer_costs.get(gw, 0)

        lineup = next((a for a in actions if isinstance(a, SetLineup)), None)
        cap_a = next((a for a in actions if isinstance(a, SetCaptain)), None)
        vc_a = next((a for a in actions if isinstance(a, SetViceCaptain)), None)
        transfers = [a for a in actions if isinstance(a, Transfer)]
        chips = [a for a in actions if isinstance(a, PlayChip)]

        cap = cap_a.player_id if cap_a else None
        vc = vc_a.player_id if vc_a else None

        # Header line
        parts = [f"### GW {gw} — {pts} pts"]
        if cost:
            parts.append(f"(−{cost} hit)")
        if chips:
            parts.append(f"| Chip: {', '.join(c.chip_type.name for c in chips)}")
        md = " ".join(parts) + "\n\n"

        # Transfers
        if transfers:
            md += "**Transfers:**\n"
            for t in transfers:
                md += f"- ⬅ {_fmt(t.player_out)} → ➡ {_fmt(t.player_in)}\n"
            md += "\n"

        # Starting XI grouped by position
        if lineup:
            md += f"**Starting XI ({_formation(lineup.starting_xi)}):**  \n"
            md += _group_by_pos(lineup.starting_xi, cap, vc) + "\n\n"
            md += f"**Bench:** {', '.join(_fmt(c) for c in lineup.bench_order)}\n"

        md += "\n---\n"
        display(Markdown(md))

interact(show_model_gws, model_name=Dropdown(options=sorted(results.keys()), description="Model:"))